# StreamGuard — Final 100% Deployment Model
## XGBoost + Magic/Behavioural Features

This notebook is **not another evaluation notebook**.

Your final evaluation is already frozen:

```text
Final untouched test PR-AUC  = 0.5854639900883427
Final untouched test ROC-AUC = 0.915749033118946
Precision                    = 0.7138657520823126
Recall                       = 0.4725916315277327
F1                           = 0.5686963309914129
```

This notebook now:

1. loads all labelled IEEE-CIS training data,
2. rebuilds the feature-engineering state from 100% of the historical data,
3. keeps the same frozen XGBoost design,
4. loads the decision threshold selected during the 70/15/15 validation run,
5. trains the final deployment model on all labelled rows,
6. saves a deployment bundle for Kafka inference.

**Do not report a PR-AUC calculated on this 100%-trained dataset.**  
The final generalisation result remains the untouched 15% test result above.


## 1. Install dependencies


In [ ]:
%pip install -q xgboost joblib


## 2. Imports


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb


## 3. Paths

Expected project layout:

```text
project/
├── data/
│   ├── train_transaction.csv
│   └── train_identity.csv
└── models/
```

The previous 70/15/15 notebook should have saved:

```text
models/streamguard_xgb_magic_70_15_15_eval_artifacts.joblib
```

That file contains the threshold chosen from validation.


In [ ]:
DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRANSACTION_FILE = DATA_DIR / "train_transaction.csv"
IDENTITY_FILE = DATA_DIR / "train_identity.csv"

EVAL_ARTIFACT_PATH = (
    MODEL_DIR
    / "streamguard_xgb_magic_70_15_15_eval_artifacts.joblib"
)

FINAL_BUNDLE_PATH = (
    MODEL_DIR
    / "streamguard_fraud_model.joblib"
)

FINAL_XGB_JSON_PATH = (
    MODEL_DIR
    / "streamguard_fraud_model.json"
)

print("Transaction:", TRANSACTION_FILE.resolve())
print("Identity:", IDENTITY_FILE.resolve())
print("Evaluation artifact:", EVAL_ARTIFACT_PATH.resolve())
print("Final bundle:", FINAL_BUNDLE_PATH.resolve())


## 4. Load the frozen validation threshold

The threshold must come from the **validation set**, not from the final test set and not from the 100% training data.

If you still have the evaluation artifact from the 70/15/15 notebook, this cell loads it automatically.

If not, paste the threshold printed by that notebook into `MANUAL_THRESHOLD`.


In [ ]:
MANUAL_THRESHOLD = None
# Example only:
# MANUAL_THRESHOLD = 0.37

eval_artifacts = None

if EVAL_ARTIFACT_PATH.exists():

    eval_artifacts = joblib.load(
        EVAL_ARTIFACT_PATH
    )

    BEST_THRESHOLD = float(
        eval_artifacts["best_threshold"]
    )

    print(
        "Loaded threshold from evaluation artifact:",
        BEST_THRESHOLD
    )

else:

    if MANUAL_THRESHOLD is None:
        raise FileNotFoundError(
            "Evaluation artifact not found. "
            "Set MANUAL_THRESHOLD to the threshold selected "
            "by the 70/15/15 validation notebook."
        )

    BEST_THRESHOLD = float(
        MANUAL_THRESHOLD
    )

    print(
        "Using manually supplied frozen threshold:",
        BEST_THRESHOLD
    )


## 5. Load and merge the complete labelled dataset


In [ ]:
transaction = pd.read_csv(
    TRANSACTION_FILE
)

identity = pd.read_csv(
    IDENTITY_FILE
)

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left",
)

# Keep chronological ordering.
df = df.sort_values(
    "TransactionDT"
).reset_index(drop=True)

print("Merged shape:", df.shape)
print("Fraud rows:", int(df["isFraud"].sum()))
print("Fraud rate:", df["isFraud"].mean())


## 6. Freeze the selected V-feature subset

This is the same selected V subset used during evaluation.


In [ ]:
v_numbers = [
    1,3,4,6,8,11,
    13,14,17,20,23,26,27,30,
    36,37,40,41,44,47,48,
    54,56,59,62,65,67,68,70,
    76,78,80,82,86,88,89,91,
    107,108,111,115,117,120,121,123,
    124,127,129,130,136,
    138,139,142,147,156,162,165,160,166,
    178,176,173,182,
    187,203,205,207,215,
    169,171,175,180,185,188,198,210,209,
    218,223,224,226,228,229,235,
    240,258,257,253,252,260,261,
    264,266,267,274,277,
    220,221,234,238,250,271,
    294,284,285,286,291,297,
    303,305,307,309,310,320,
    281,283,289,296,301,314,
]

selected_v_cols = [
    f"V{i}"
    for i in v_numbers
    if f"V{i}" in df.columns
]

print("Selected V features:", len(selected_v_cols))


## 7. Keep the same raw feature families


In [ ]:
base_cols = [
    "TransactionID",
    "TransactionDT",
    "TransactionAmt",
    "isFraud",
    "ProductCD",
    "card1", "card2", "card3",
    "card4", "card5", "card6",
    "addr1", "addr2",
    "dist1", "dist2",
    "P_emaildomain",
    "R_emaildomain",
]

base_cols += [
    f"C{i}"
    for i in range(1, 15)
    if f"C{i}" in df.columns
]

base_cols += [
    f"D{i}"
    for i in range(1, 16)
    if f"D{i}" in df.columns
]

base_cols += [
    f"M{i}"
    for i in range(1, 10)
    if f"M{i}" in df.columns
]

identity_cols = [
    column
    for column in df.columns
    if column.startswith("id_")
    or column in [
        "DeviceType",
        "DeviceInfo",
    ]
]

keep_cols = []
seen = set()

for column in (
    base_cols
    + selected_v_cols
    + identity_cols
):

    if (
        column in df.columns
        and column not in seen
    ):
        keep_cols.append(
            column
        )

        seen.add(
            column
        )

full = df[
    keep_cols
].copy()

del df

print(
    "100% training working shape:",
    full.shape
)


## 8. Row-local features

These use only information present in the transaction itself.


In [ ]:
full["cents"] = (
    full["TransactionAmt"]
    - np.floor(
        full["TransactionAmt"]
    )
).astype("float32")

full["day"] = (
    full["TransactionDT"]
    / 86400.0
).astype("float32")

full["DT_M"] = (
    full["TransactionDT"]
    // (86400 * 30)
).astype("int16")

full["tx_hour"] = (
    (
        full["TransactionDT"]
        // 3600
    ) % 24
).astype("int8")

full["tx_dow"] = (
    (
        full["TransactionDT"]
        // 86400
    ) % 7
).astype("int8")

full["amt_log1p"] = np.log1p(
    full["TransactionAmt"]
).astype("float32")


## 9. Normalize D columns


In [ ]:
for i in range(1, 16):

    if i in [
        1, 2, 3, 5, 9
    ]:
        continue

    column = f"D{i}"

    if column in full.columns:

        full[column] = (
            full[column]
            - full["TransactionDT"]
            / 86400.0
        ).astype("float32")


## 10. Build the pseudo-customer keys

These keys are used to build behavioural statistics.

The raw `uid` itself is not sent to XGBoost.


In [ ]:
full["card1_addr1"] = (
    full["card1"].astype(str)
    + "_"
    + full["addr1"].astype(str)
)

full[
    "card1_addr1_P_emaildomain"
] = (
    full["card1_addr1"]
    + "_"
    + full[
        "P_emaildomain"
    ].astype(str)
)

full["uid"] = (
    full["card1_addr1"]
    + "_"
    + np.floor(
        full["day"]
        - full["D1"]
    ).astype(str)
)

full[[
    "card1",
    "addr1",
    "D1",
    "day",
    "uid",
]].head()


## 11. Fit frequency maps on all historical labelled data

This is correct for deployment: at the deployment cutoff, all of these rows are now historical data.


In [ ]:
def fit_frequency_map(
    series
):

    normalized = (
        series
        .astype("object")
        .where(
            series.notna(),
            "__MISSING__",
        )
        .astype(str)
    )

    return (
        normalized
        .value_counts(
            normalize=True,
            dropna=False,
        )
        .to_dict()
    )


def apply_frequency_map(
    series,
    mapping,
):

    normalized = (
        series
        .astype("object")
        .where(
            series.notna(),
            "__MISSING__",
        )
        .astype(str)
    )

    return (
        normalized
        .map(mapping)
        .fillna(0)
        .astype("float32")
    )


In [ ]:
frequency_columns = [
    "addr1",
    "card1",
    "card2",
    "card3",
    "P_emaildomain",
    "card1_addr1",
    "card1_addr1_P_emaildomain",
    "uid",
]

frequency_maps = {}

for column in frequency_columns:

    if column not in full.columns:
        continue

    mapping = fit_frequency_map(
        full[column]
    )

    frequency_maps[
        column
    ] = mapping

    full[
        f"{column}_FE"
    ] = apply_frequency_map(
        full[column],
        mapping,
    )

print(
    "Frequency maps:",
    len(frequency_maps)
)


## 12. Group mean/std maps


In [ ]:
group_stat_maps = {}

def add_group_mean_std(
    frame,
    group_column,
    value_column,
):

    stats = (
        frame
        .groupby(
            group_column
        )[value_column]
        .agg([
            "mean",
            "std",
        ])
    )

    mean_map = (
        stats["mean"]
        .to_dict()
    )

    std_map = (
        stats["std"]
        .to_dict()
    )

    group_stat_maps[
        (
            group_column,
            value_column,
            "mean",
        )
    ] = mean_map

    group_stat_maps[
        (
            group_column,
            value_column,
            "std",
        )
    ] = std_map

    frame[
        f"{value_column}_{group_column}_mean"
    ] = (
        frame[group_column]
        .map(mean_map)
        .astype("float32")
    )

    frame[
        f"{value_column}_{group_column}_std"
    ] = (
        frame[group_column]
        .map(std_map)
        .astype("float32")
    )


## 13. Card/address behavioural aggregates


In [ ]:
group_columns = [
    "card1",
    "card1_addr1",
    "card1_addr1_P_emaildomain",
]

aggregate_values = [
    "TransactionAmt",
    "D9",
    "D11",
]

for group_column in group_columns:

    for value_column in aggregate_values:

        if (
            group_column
            in full.columns
            and value_column
            in full.columns
        ):

            add_group_mean_std(
                full,
                group_column,
                value_column,
            )


## 14. UID behavioural mean/std features


In [ ]:
for value_column in [
    "TransactionAmt",
    "D4",
    "D9",
    "D10",
    "D15",
]:

    if value_column in full.columns:

        add_group_mean_std(
            full,
            "uid",
            value_column,
        )


## 15. UID means for C features


In [ ]:
uid_mean_maps = {}

for value_column in [
    f"C{i}"
    for i in range(1, 15)
    if i != 3
]:

    if value_column not in full.columns:
        continue

    mapping = (
        full
        .groupby("uid")[
            value_column
        ]
        .mean()
        .to_dict()
    )

    uid_mean_maps[
        value_column
    ] = mapping

    full[
        f"{value_column}_uid_mean"
    ] = (
        full["uid"]
        .map(mapping)
        .astype("float32")
    )


## 16. Convert M features and create UID means


In [ ]:
m_numeric_cols = []

for i in range(1, 10):

    column = f"M{i}"

    if column not in full.columns:
        continue

    numeric_column = (
        f"{column}_numeric"
    )

    true_false_map = {
        "T": 1.0,
        "F": 0.0,
        "True": 1.0,
        "False": 0.0,
    }

    full[
        numeric_column
    ] = (
        full[column]
        .astype(str)
        .map(
            true_false_map
        )
        .astype("float32")
    )

    m_numeric_cols.append(
        numeric_column
    )

uid_m_mean_maps = {}

for value_column in m_numeric_cols:

    mapping = (
        full
        .groupby("uid")[
            value_column
        ]
        .mean()
        .to_dict()
    )

    uid_m_mean_maps[
        value_column
    ] = mapping

    full[
        f"{value_column}_uid_mean"
    ] = (
        full["uid"]
        .map(mapping)
        .astype("float32")
    )

print(
    "M-derived features:",
    len(m_numeric_cols)
)


## 17. UID unique-count features


In [ ]:
uid_nunique_maps = {}

uid_nunique_columns = [
    "P_emaildomain",
    "dist1",
    "DT_M",
    "id_02",
    "cents",
    "C13",
    "V314",
    "V127",
    "V136",
    "V309",
    "V307",
    "V320",
]

for value_column in uid_nunique_columns:

    if value_column not in full.columns:
        continue

    mapping = (
        full
        .groupby("uid")[
            value_column
        ]
        .nunique()
        .to_dict()
    )

    uid_nunique_maps[
        value_column
    ] = mapping

    full[
        f"uid_{value_column}_ct"
    ] = (
        full["uid"]
        .map(mapping)
        .fillna(0)
        .astype("float32")
    )


## 18. outsider15 feature


In [ ]:
if (
    "D1" in full.columns
    and "D15" in full.columns
):

    full[
        "outsider15"
    ] = (
        np.abs(
            full["D1"]
            - full["D15"]
        ) > 3
    ).astype("int8")


## 19. Fit categorical mappings on all historical labelled data

These mappings are saved for future Kafka transactions.

Any unseen category at inference time should map to `-1`.


In [ ]:
category_maps = {}

categorical_columns = (
    full
    .select_dtypes(
        include=[
            "object",
            "category",
            "string",
        ]
    )
    .columns
    .tolist()
)

for column in categorical_columns:

    if column == "uid":
        continue

    values = (
        full[column]
        .astype("object")
        .where(
            full[column].notna(),
            "__MISSING__",
        )
        .astype(str)
    )

    unique_values = (
        values
        .drop_duplicates()
        .tolist()
    )

    mapping = {
        value: index
        for index, value
        in enumerate(
            unique_values
        )
    }

    category_maps[
        column
    ] = mapping

    full[column] = (
        values
        .map(mapping)
        .fillna(-1)
        .astype("int32")
    )

print(
    "Categorical maps:",
    len(category_maps)
)


## 20. Freeze feature order

If the previous evaluation artifact is available, reuse its feature list exactly.

This prevents accidental feature-order changes between evaluation and deployment.


In [ ]:
exclude = {
    "isFraud",
    "TransactionID",
    "TransactionDT",
    "uid",
    "day",
}

generated_features = [
    column
    for column in full.columns
    if (
        column not in exclude
        and pd.api.types.is_numeric_dtype(
            full[column]
        )
    )
]

if (
    eval_artifacts is not None
    and "features"
    in eval_artifacts
):

    missing = [
        column
        for column
        in eval_artifacts["features"]
        if column not in full.columns
    ]

    if missing:
        raise ValueError(
            "Deployment dataframe is missing "
            f"evaluation features: {missing[:10]}"
        )

    features = list(
        eval_artifacts[
            "features"
        ]
    )

    print(
        "Using frozen feature order "
        "from evaluation artifact."
    )

else:

    features = (
        generated_features
    )

    print(
        "Evaluation feature list unavailable; "
        "using generated feature order."
    )

print(
    "Final feature count:",
    len(features)
)

print(
    "Training matrix:",
    full[features].shape
)


## 21. Frozen XGBoost configuration

Do not tune this configuration using the final test result.


In [ ]:
final_model = xgb.XGBClassifier(
    objective="binary:logistic",

    n_estimators=2000,
    max_depth=12,
    learning_rate=0.02,

    subsample=0.8,
    colsample_bytree=0.4,

    eval_metric="aucpr",
    tree_method="hist",

    random_state=42,
    n_jobs=-1,
)

print(
    "XGBoost version:",
    xgb.__version__
)

# Optional NVIDIA GPU:
# final_model.set_params(
#     device="cuda"
# )


# 22. Train final model on 100%

There is no validation/test set here because evaluation has already been completed.

This model is being prepared for deployment.


In [ ]:
X_full = full[
    features
]

y_full = full[
    "isFraud"
]

final_model.fit(
    X_full,
    y_full,
    verbose=False,
)

print(
    "Final 100% model trained."
)


## 23. Do NOT calculate deployment PR-AUC on the training rows

A score calculated here would be an in-sample training score and should not replace your final untouched test result.

Your report remains:

```text
Final Test PR-AUC  = 0.5854639900883427
Final Test ROC-AUC = 0.915749033118946
```


## 24. Save the final deployment bundle

The bundle contains:

- trained XGBoost model
- exact feature order
- frozen validation threshold
- frequency maps
- category maps
- group statistics
- UID statistics
- feature configuration
- final evaluation metadata
- model version


In [ ]:
deployment_bundle = {
    "model": final_model,

    "model_version": (
        "xgb-magic-v1"
    ),

    "features": features,

    "threshold": (
        BEST_THRESHOLD
    ),

    "selected_v_cols": (
        selected_v_cols
    ),

    "frequency_maps": (
        frequency_maps
    ),

    "category_maps": (
        category_maps
    ),

    "group_stat_maps": (
        group_stat_maps
    ),

    "uid_mean_maps": (
        uid_mean_maps
    ),

    "uid_m_mean_maps": (
        uid_m_mean_maps
    ),

    "uid_nunique_maps": (
        uid_nunique_maps
    ),

    "m_numeric_cols": (
        m_numeric_cols
    ),

    "evaluation": {
        "test_pr_auc": (
            0.5854639900883427
        ),
        "test_roc_auc": (
            0.915749033118946
        ),
        "test_precision": (
            0.7138657520823126
        ),
        "test_recall": (
            0.4725916315277327
        ),
        "test_f1": (
            0.5686963309914129
        ),
    },
}

joblib.dump(
    deployment_bundle,
    FINAL_BUNDLE_PATH,
)

final_model.save_model(
    FINAL_XGB_JSON_PATH
)

print(
    "Saved deployment bundle:",
    FINAL_BUNDLE_PATH.resolve()
)

print(
    "Saved XGBoost JSON:",
    FINAL_XGB_JSON_PATH.resolve()
)


## 25. Reload smoke test

Always verify that the saved bundle can be loaded before using it in Kafka.


In [ ]:
loaded_bundle = joblib.load(
    FINAL_BUNDLE_PATH
)

loaded_model = loaded_bundle[
    "model"
]

loaded_features = loaded_bundle[
    "features"
]

loaded_threshold = loaded_bundle[
    "threshold"
]

sample = full[
    loaded_features
].tail(5)

sample_probabilities = (
    loaded_model
    .predict_proba(
        sample
    )[:, 1]
)

sample_predictions = (
    sample_probabilities
    >= loaded_threshold
).astype(int)

print(
    "Model version:",
    loaded_bundle[
        "model_version"
    ]
)

print(
    "Threshold:",
    loaded_threshold
)

print(
    "Probabilities:",
    sample_probabilities
)

print(
    "Predictions:",
    sample_predictions
)


## 26. Deployment output contract

Your Kafka ML consumer should eventually emit something like:

```json
{
  "transaction_id": 123456,
  "fraud_probability": 0.83,
  "fraud_prediction": 1,
  "threshold": 0.37,
  "model_version": "xgb-magic-v1"
}
```

The next development step is to build a **streaming feature-transform function** that:

```text
raw Kafka transaction
        ↓
row-local features
        ↓
UID construction
        ↓
saved historical maps
        ↓
exact model feature vector
        ↓
predict_proba()
        ↓
fraud-results Kafka topic
```

That transformer should use the maps saved in `streamguard_fraud_model.joblib`.


# Final status

At the end of this notebook you should have:

```text
models/
├── streamguard_fraud_model.joblib
└── streamguard_fraud_model.json
```

Use the `.joblib` bundle for the StreamGuard Python/Kafka consumer because it contains both the model and its preprocessing state.

Your ML evaluation remains frozen at:

```text
PR-AUC  = 0.5854639900883427
ROC-AUC = 0.915749033118946
```

After this notebook, the next task is **Kafka inference integration**, not more model tuning.
